In [128]:
"""
Hyperparameter Importance Analysis for TDE and FastSHAP
Generates bar plots showing average feature importance across all datasets.

Author: Fati
Date: 2025
"""

# ============================================================================
# LIBRARY IMPORTS
# ============================================================================
import os
import sqlite3
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

import optuna
from optuna.importance import get_param_importances

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION - EASY TO MODIFY
# ============================================================================

# Database paths
EXPLAINER_DB = "explainer_results.db"
BENCHMARK_DB = "benchmark_results.db"
RESULTS_BASE_DIR = "results"

# Output settings
OUTPUT_DIR = "results/hyperparameter_importance"
OUTPUT_FORMAT = "pdf"  # 'pdf' or 'png'
DPI = 300

# Font settings - SINGLE FONT SIZE FOR EVERYTHING
BASE_FONT_SIZE = 22
FONT_FAMILY = 'sans-serif'

# Figure settings
FIGURE_WIDTH = 14
FIGURE_HEIGHT = 8
BAR_WIDTH = 0.7  # Increased from 0.5
BAR_ALPHA = 0.85
BAR_EDGE_COLOR = 'black'
BAR_EDGE_WIDTH = 0.8

# Color settings
TDE_COLOR = '#2ecc71'        # Green
FASTSHAP_COLOR = '#3498db'   # Blue
GRID_ALPHA = 0.3
GRID_STYLE = '--'

# Plot style settings
LABEL_PAD = 10
ROTATION_ANGLE = 45          # X-axis label rotation

# Toggle settings
SHOW_BAR_ANNOTATIONS = False  # Set to False to hide text on top of bars
SHOW_ERROR_BARS = False       # Set to False to hide error bars
SHOW_X_AXIS_LABEL = False     # Set to False to hide "Hyperparameter" label
SHOW_Y_AXIS_LABEL = True      # Set to False to hide "Importance Score" label
SHOW_TITLE = False            # Set to False to hide title
SHOW_LEGEND = False           # Set to False to hide legend

# Hyperparameter name formatting for publication
HYPERPARAMETER_DISPLAY_NAMES = {
    'l1_lambda': 'L1 Regularization (λ₁)',
    'l2_lambda': 'L2 Regularization (λ₂)',
    'smoothness_lambda': 'Smoothness (λₛ)',
    'efficiency_lambda': 'Efficiency (λₑ)',
    'sparsity_lambda': 'Sparsity (λₚ)',
    'target_sparsity': 'Target Sparsity',
    'sparsity_threshold': 'Sparsity Threshold',
    'hidden_dim': 'Hidden Dimension',
    'n_conv_layers': 'Conv Layers',
    'n_layers': 'Hidden Layers',
    'kernel_size': 'Kernel Size',
    'n_attention_heads': 'Attention Heads',
    'dropout_rate': 'Dropout Rate',
    'batch_size': 'Batch Size',
    'learning_rate': 'Learning Rate',
    'optimizer_type': 'Optimizer Type',
    'masking_mode': 'Masking Mode',
    'samples_per_feature': 'Samples per Feature',
    'window_size': 'Window Size',
    'weight_decay': 'Weight Decay',
}

# Importance calculation settings
MIN_TRIALS_FOR_IMPORTANCE = 3  # Minimum trials needed to calculate importance
TOP_N_HYPERPARAMETERS = 15     # Maximum number of hyperparameters to show

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def setup_plot_style():
    """Setup matplotlib style for consistent plots."""
    plt.rcParams.update({
        'font.size': BASE_FONT_SIZE,
        'font.family': FONT_FAMILY,
        'axes.titlesize': BASE_FONT_SIZE,
        'axes.labelsize': BASE_FONT_SIZE,
        'xtick.labelsize': BASE_FONT_SIZE,
        'ytick.labelsize': BASE_FONT_SIZE,
        'legend.fontsize': BASE_FONT_SIZE,
        'figure.titlesize': BASE_FONT_SIZE,
        'axes.grid': True,
        'grid.alpha': GRID_ALPHA,
        'grid.linestyle': GRID_STYLE,
    })

def get_all_datasets():
    """Get all available datasets from benchmark database."""
    if not os.path.exists(BENCHMARK_DB):
        print(f"⚠️ Benchmark database not found: {BENCHMARK_DB}")
        return []
    
    conn = sqlite3.connect(BENCHMARK_DB)
    query = '''
        SELECT DISTINCT primary_use, option_number 
        FROM prediction_performance 
        ORDER BY primary_use, option_number
    '''
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    return [{'primary_use': r['primary_use'], 'option_number': int(r['option_number'])} 
            for _, r in df.iterrows()]

def get_models_for_dataset(primary_use, option_number):
    """Get all models for a specific dataset."""
    conn = sqlite3.connect(BENCHMARK_DB)
    query = '''
        SELECT DISTINCT model_name 
        FROM prediction_performance 
        WHERE primary_use = ? AND option_number = ?
    '''
    df = pd.read_sql_query(query, conn, params=(primary_use, option_number))
    conn.close()
    return df['model_name'].tolist()

def get_optuna_study_path(primary_use, option_number, model_name, explainer_type):
    """Get the path to Optuna study database."""
    exp_type = explainer_type.lower()
    path = Path(RESULTS_BASE_DIR) / primary_use / f"option_{option_number}" / model_name.lower() / exp_type / "optuna_study.db"
    return str(path)

def load_optuna_study(study_path, study_name):
    """Load Optuna study from database."""
    if not os.path.exists(study_path):
        return None
    
    try:
        storage = f"sqlite:///{study_path}"
        study = optuna.load_study(study_name=study_name, storage=storage)
        return study
    except Exception as e:
        return None

def get_hyperparameter_importance(study):
    """Get hyperparameter importance from Optuna study."""
    if study is None or len(study.trials) < MIN_TRIALS_FOR_IMPORTANCE:
        return None
    
    try:
        # Filter completed trials
        completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if len(completed_trials) < MIN_TRIALS_FOR_IMPORTANCE:
            return None
        
        importance = get_param_importances(study)
        return importance
    except Exception as e:
        return None

def collect_all_importances(explainer_type):
    """
    Collect hyperparameter importances across all datasets and models.
    
    Returns:
        dict: {hyperparameter_name: [list of importance values]}
    """
    all_importances = {}
    datasets = get_all_datasets()
    
    if not datasets:
        print(f"⚠️ No datasets found!")
        return all_importances
    
    exp_type_lower = explainer_type.lower()
    study_prefix = f"{exp_type_lower}_"
    
    print(f"\n📊 Collecting {explainer_type.upper()} hyperparameter importances...")
    print(f"   Found {len(datasets)} datasets")
    
    total_studies = 0
    successful_studies = 0
    
    for ds in datasets:
        primary_use = ds['primary_use']
        option_number = ds['option_number']
        
        models = get_models_for_dataset(primary_use, option_number)
        
        for model_name in models:
            total_studies += 1
            study_path = get_optuna_study_path(primary_use, option_number, model_name, exp_type_lower)
            study_name = f"{exp_type_lower}_{model_name}"
            
            study = load_optuna_study(study_path, study_name)
            
            if study is None:
                continue
            
            importance = get_hyperparameter_importance(study)
            
            if importance is None:
                continue
            
            successful_studies += 1
            
            # Aggregate importances
            for param, imp_value in importance.items():
                if param not in all_importances:
                    all_importances[param] = []
                all_importances[param].append(imp_value)
    
    print(f"   Processed {successful_studies}/{total_studies} studies successfully")
    
    return all_importances

def compute_average_importance(all_importances):
    """
    Compute average importance for each hyperparameter.
    
    Returns:
        dict: {hyperparameter_name: {'mean': float, 'std': float, 'count': int}}
    """
    avg_importance = {}
    
    for param, values in all_importances.items():
        if len(values) > 0:
            avg_importance[param] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'count': len(values)
            }
    
    # Sort by mean importance (descending)
    avg_importance = dict(sorted(avg_importance.items(), 
                                  key=lambda x: x[1]['mean'], 
                                  reverse=True))
    
    return avg_importance

def format_param_name(param_name):
    """Convert parameter name to publication-ready format."""
    if param_name in HYPERPARAMETER_DISPLAY_NAMES:
        return HYPERPARAMETER_DISPLAY_NAMES[param_name]
    # Fallback: capitalize and replace underscores with spaces
    return param_name.replace('_', ' ').title()

def save_importance_to_log(avg_importance, explainer_type, log_path):
    """Save hyperparameter importance data to log file."""
    params = list(avg_importance.keys())[:TOP_N_HYPERPARAMETERS]
    display_names = [format_param_name(p) for p in params]
    means = [avg_importance[p]['mean'] for p in params]
    
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(f"{explainer_type.upper()}\n")
        f.write(f"Hyperparameter\tImportance\n")
        for display, mean in zip(display_names, means):
            f.write(f"{display}\t{mean:.6f}\n")
        f.write("\n")

def create_single_explainer_plot(avg_importance, explainer_type, output_path):
    """Create bar plot for a single explainer type."""
    if not avg_importance:
        print(f"⚠️ No importance data for {explainer_type}")
        return False
    
    setup_plot_style()
    
    # Limit to top N hyperparameters
    params = list(avg_importance.keys())[:TOP_N_HYPERPARAMETERS]
    display_names = [format_param_name(p) for p in params]
    means = [avg_importance[p]['mean'] for p in params]
    stds = [avg_importance[p]['std'] for p in params]
    counts = [avg_importance[p]['count'] for p in params]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
    
    # Create bars
    x = np.arange(len(params))
    color = TDE_COLOR if explainer_type.lower() == 'tde' else FASTSHAP_COLOR
    
    # Conditionally add error bars
    if SHOW_ERROR_BARS:
        bars = ax.bar(x, means, BAR_WIDTH, yerr=stds, capsize=4,
                      color=color, alpha=BAR_ALPHA,
                      edgecolor=BAR_EDGE_COLOR, linewidth=BAR_EDGE_WIDTH)
    else:
        bars = ax.bar(x, means, BAR_WIDTH,
                      color=color, alpha=BAR_ALPHA,
                      edgecolor=BAR_EDGE_COLOR, linewidth=BAR_EDGE_WIDTH)
    
    # Conditionally add value annotations on bars
    if SHOW_BAR_ANNOTATIONS:
        for i, (bar, mean, std, count) in enumerate(zip(bars, means, stds, counts)):
            height = bar.get_height()
            annotation_y = height + std if SHOW_ERROR_BARS else height
            ax.annotate(f'{mean:.3f}\n(n={count})',
                        xy=(bar.get_x() + bar.get_width() / 2, annotation_y),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom',
                        fontsize=BASE_FONT_SIZE,
                        fontweight='bold')
    
    # Formatting - conditionally show axis labels
    if SHOW_X_AXIS_LABEL:
        ax.set_xlabel('Hyperparameter', fontsize=BASE_FONT_SIZE, labelpad=LABEL_PAD)
    
    if SHOW_Y_AXIS_LABEL:
        ax.set_ylabel('Importance Score', fontsize=BASE_FONT_SIZE, labelpad=LABEL_PAD)
    
    # Conditionally show title
    if SHOW_TITLE:
        ax.set_title(f'{explainer_type.upper()} Hyperparameter Importance\n(Average Across All Datasets & Models)',
                     fontsize=BASE_FONT_SIZE, fontweight='bold', pad=20)
    
    ax.set_xticks(x)
    ax.set_xticklabels(display_names, rotation=ROTATION_ANGLE, ha='right', fontsize=BASE_FONT_SIZE)
    
    # Conditionally show legend
    if SHOW_LEGEND:
        ax.legend([f'{explainer_type.upper()} (n={sum(counts)})'], 
                  loc='upper right', fontsize=BASE_FONT_SIZE)
    
    ax.set_axisbelow(True)
    ax.grid(True, alpha=GRID_ALPHA, linestyle=GRID_STYLE, axis='y')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save
    plt.savefig(output_path, format=OUTPUT_FORMAT, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    
    print(f"✅ Saved: {output_path}")
    return True


def print_summary(tde_importance, fastshap_importance):
    """Print summary statistics to console."""
    print("\n" + "=" * 80)
    print("📊 HYPERPARAMETER IMPORTANCE SUMMARY")
    print("=" * 80)
    
    print("\n🟢 TDE Top Hyperparameters:")
    print("-" * 50)
    for i, (param, data) in enumerate(list(tde_importance.items())[:10], 1):
        print(f"  {i:2d}. {param:<25} Mean: {data['mean']:.4f} ± {data['std']:.4f} (n={data['count']})")
    
    print("\n🔵 FastSHAP Top Hyperparameters:")
    print("-" * 50)
    for i, (param, data) in enumerate(list(fastshap_importance.items())[:10], 1):
        print(f"  {i:2d}. {param:<25} Mean: {data['mean']:.4f} ± {data['std']:.4f} (n={data['count']})")
    
    print("\n" + "=" * 80)

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def main():
    """Main function to generate hyperparameter importance plots."""
    print("\n" + "=" * 80)
    print("🚀 HYPERPARAMETER IMPORTANCE ANALYSIS")
    print("=" * 80)
    print(f"📁 Output Directory: {OUTPUT_DIR}")
    print(f"📄 Output Format: {OUTPUT_FORMAT.upper()}")
    print(f"🔧 DPI: {DPI}")
    print("=" * 80)
    
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Collect importances for TDE
    tde_raw_importance = collect_all_importances('tde')
    tde_avg_importance = compute_average_importance(tde_raw_importance)
    
    # Collect importances for FastSHAP
    fastshap_raw_importance = collect_all_importances('fastshap')
    fastshap_avg_importance = compute_average_importance(fastshap_raw_importance)
    
    # Check if we have data
    if not tde_avg_importance and not fastshap_avg_importance:
        print("\n⚠️ No hyperparameter importance data found!")
        print("   Make sure you have trained TDE/FastSHAP models with Optuna optimization.")
        return
    
    # Print summary
    print_summary(tde_avg_importance, fastshap_avg_importance)
    
    # Define static output filenames
    log_output = os.path.join(OUTPUT_DIR, f"hyperparameter_importance.log")
    
    # Clear log file if it exists (start fresh)
    if os.path.exists(log_output):
        os.remove(log_output)
    
    # Create TDE plot and save to log
    if tde_avg_importance:
        tde_output = os.path.join(OUTPUT_DIR, f"tde_hyperparameter_importance.{OUTPUT_FORMAT}")
        create_single_explainer_plot(tde_avg_importance, 'TDE', tde_output)
        save_importance_to_log(tde_avg_importance, 'TDE', log_output)
    
    # Create FastSHAP plot and save to log
    if fastshap_avg_importance:
        fastshap_output = os.path.join(OUTPUT_DIR, f"fastshap_hyperparameter_importance.{OUTPUT_FORMAT}")
        create_single_explainer_plot(fastshap_avg_importance, 'FastSHAP', fastshap_output)
        save_importance_to_log(fastshap_avg_importance, 'FastSHAP', log_output)
    
    print("\n" + "=" * 80)
    print("✅ ANALYSIS COMPLETE")
    print(f"📁 Results saved to: {OUTPUT_DIR}")
    print(f"📊 Log file: {log_output}")
    print("=" * 80)

if __name__ == "__main__":
    main()


🚀 HYPERPARAMETER IMPORTANCE ANALYSIS
📁 Output Directory: results/hyperparameter_importance
📄 Output Format: PDF
🔧 DPI: 300

📊 Collecting TDE hyperparameter importances...
   Found 5 datasets
   Processed 50/50 studies successfully

📊 Collecting FASTSHAP hyperparameter importances...
   Found 5 datasets
   Processed 50/50 studies successfully

📊 HYPERPARAMETER IMPORTANCE SUMMARY

🟢 TDE Top Hyperparameters:
--------------------------------------------------
   1. sparsity_threshold        Mean: 0.2099 ± 0.2282 (n=50)
   2. target_sparsity           Mean: 0.1175 ± 0.0709 (n=50)
   3. kernel_size               Mean: 0.1120 ± 0.0724 (n=50)
   4. smoothness_lambda         Mean: 0.0802 ± 0.0833 (n=50)
   5. learning_rate             Mean: 0.0540 ± 0.0513 (n=50)
   6. window_size               Mean: 0.0506 ± 0.0372 (n=50)
   7. efficiency_lambda         Mean: 0.0453 ± 0.0317 (n=50)
   8. dropout_rate              Mean: 0.0442 ± 0.0313 (n=50)
   9. optimizer_type            Mean: 0.0422 ± 0.03

# Metrics bar plots

In [13]:
# ============================
# LIBRARY IMPORTS
# ============================
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ============================
# CONFIGURATION - USER ADJUSTABLE
# ============================
# Database
XAI_DB = "databases/xai_results.db"
OUTPUT_DIR = "./results/metrics"

# Plot dimensions
FIGURE_WIDTH = 22
FIGURE_HEIGHT = 8
LEGEND_FIGURE_WIDTH = 12
LEGEND_FIGURE_HEIGHT = 2

# Font sizes - USER ADJUSTABLE
TITLE_FONT_SIZE = 38
AXIS_LABEL_FONT_SIZE = TITLE_FONT_SIZE
X_TICK_FONT_SIZE = TITLE_FONT_SIZE
Y_TICK_FONT_SIZE = TITLE_FONT_SIZE
LEGEND_FONT_SIZE = TITLE_FONT_SIZE

# Display options (True/False toggles)
SHOW_TITLE = False
SHOW_X_AXIS_LABEL = False
SHOW_Y_AXIS_LABEL = True
SHOW_LEGEND = False
SHOW_GRID = True
SHOW_VALUE_ANNOTATIONS = False
SHOW_ERROR_BARS = False

# Bar styling
BAR_WIDTH = 0.11
BAR_EDGE_COLOR = 'black'
BAR_EDGE_WIDTH = 1.5
BAR_ALPHA = 0.85

# X-axis margins (smaller = less empty space, bars appear wider)
X_MARGIN_LEFT = 0.5
X_MARGIN_RIGHT = 0.5

# Grid styling
GRID_ALPHA = 0.3
GRID_AXIS = 'y'

# Color scheme for XAI methods
METHOD_COLORS = {
    'tde': '#2ecc71',
    'fastshap': '#3498db',
    'gradient': '#e74c3c',
    'deep': '#9b59b6',
    'permutation': '#f39c12',
    'kernel': '#1abc9c',
    'partition': '#e91e63',
    'lime': '#795548',
    'sampling': '#607d8b'
}

# Primary Use Label Mapping - USER ADJUSTABLE
# Format: 'database_value': 'Display\nLabel' (use \n for line breaks)
PRIMARY_USE_LABELS = {
    'residential': 'Residential',
    'health': 'Medical\nClinic',
    'warehouse': 'Manufacturing\nFacility',
    'office': 'Office\nBuilding',
    'retail': 'Retail\nStore'
}

# Metrics configuration - ALL 9 METRICS
METRICS_CONFIG = {
    'fidelity': {
        'column': 'fidelity',
        'title': 'Fidelity Score by Primary Use and XAI Method',
        'ylabel': 'Fidelity (Higher = Better)',
        'higher_better': True,
        'format': '.4f',
        'log_scale': False
    },
    'sparsity': {
        'column': 'sparsity',
        'title': 'Sparsity by Primary Use and XAI Method',
        'ylabel': 'Sparsity %',
        'higher_better': True,
        'format': '.1f',
        'log_scale': False
    },
    'complexity': {
        'column': 'complexity',
        'title': 'Complexity (Entropy) by Primary Use and XAI Method',
        'ylabel': 'Entropy (Lower = Better)',
        'higher_better': False,
        'format': '.3f',
        'log_scale': False
    },
    'reliability_correlation': {
        'column': 'reliability_correlation',
        'title': 'Reliability (Pearson Correlation) by Primary Use and XAI Method',
        'ylabel': 'Correlation (Higher = Better)',
        'higher_better': True,
        'format': '.4f',
        'log_scale': False
    },
    'reliability_ped': {
        'column': 'reliability_ped',
        'title': 'Reliability (Percentage Error) by Primary Use and XAI Method',
        'ylabel': 'Log Scale Error %',
        'higher_better': False,
        'format': '.2f',
        'log_scale': True
    },
    'reliability_topk': {
        'column': 'reliability_topk_overlap',
        'title': 'Reliability (Top-K Overlap) by Primary Use and XAI Method',
        'ylabel': 'Top-K Overlap %',
        'higher_better': True,
        'format': '.1f',
        'log_scale': False
    },
    'reliability_kendall': {
        'column': 'reliability_kendall_tau',
        'title': 'Reliability (Kendall Tau) by Primary Use and XAI Method',
        'ylabel': 'Kendall Tau (Higher = Better)',
        'higher_better': True,
        'format': '.4f',
        'log_scale': False
    },
    'efficiency_error': {
        'column': 'efficiency_error',
        'title': 'Efficiency Error by Primary Use and XAI Method',
        'ylabel': 'Relative Error',
        'higher_better': False,
        'format': '.4f',
        'log_scale': False
    },
    'computation_time': {
        'column': 'computation_time',
        'title': 'Computation Time by Primary Use and XAI Method',
        'ylabel': 'Log Time',
        'higher_better': False,
        'format': '.3f',
        'log_scale': True
    }
}


# ============================
# HELPER FUNCTIONS
# ============================
def get_display_label(primary_use):
    """Get display label for a primary use value."""
    # Check if we have a custom mapping
    if primary_use.lower() in PRIMARY_USE_LABELS:
        return PRIMARY_USE_LABELS[primary_use.lower()]
    # Default: capitalize first letter
    return primary_use.capitalize()


def check_database_columns(db_path=XAI_DB):
    """Check which metric columns exist in the database."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA table_info(xai_results)")
    columns = [row[1] for row in cursor.fetchall()]
    conn.close()
    
    expected_metrics = {
        'fidelity', 'sparsity', 'complexity',
        'reliability_correlation', 'reliability_ped', 
        'reliability_topk_overlap', 'reliability_kendall_tau',
        'efficiency_error', 'computation_time'
    }
    
    existing = set(columns) & expected_metrics
    missing = expected_metrics - existing
    
    return {
        'all_columns': columns,
        'existing_metrics': sorted(existing),
        'missing_metrics': sorted(missing)
    }


# ============================
# DATA LOADING
# ============================
def load_xai_results_by_primary_use(db_path=XAI_DB):
    """Load XAI results from database aggregated by primary_use and xai_method."""
    conn = sqlite3.connect(db_path)
    
    # First, check which columns exist in the database
    cursor = conn.cursor()
    cursor.execute("PRAGMA table_info(xai_results)")
    columns = [row[1] for row in cursor.fetchall()]
    
    # Build query with only existing columns
    base_cols = ['primary_use', 'xai_method']
    metric_cols = {
        'fidelity': 'fidelity',
        'sparsity': 'sparsity', 
        'complexity': 'complexity',
        'reliability_correlation': 'reliability_correlation',
        'reliability_ped': 'reliability_ped',
        'reliability_topk_overlap': 'reliability_topk_overlap',
        'reliability_kendall_tau': 'reliability_kendall_tau',
        'efficiency_error': 'efficiency_error',
        'computation_time': 'computation_time'
    }
    
    select_parts = base_cols.copy()
    for alias, col in metric_cols.items():
        if col in columns:
            select_parts.append(f'AVG({col}) as {alias}')
    
    select_parts.extend(['COUNT(*) as n_samples', 'COUNT(DISTINCT model_name) as n_models'])
    
    query = f'''
        SELECT {', '.join(select_parts)}
        FROM xai_results
        GROUP BY primary_use, xai_method
        ORDER BY primary_use, xai_method
    '''
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


def load_xai_results(primary_use=None, option_number=None, db_path=XAI_DB):
    """Load XAI results from database with mean aggregation (legacy function)."""
    conn = sqlite3.connect(db_path)
    
    # Check which columns exist
    cursor = conn.cursor()
    cursor.execute("PRAGMA table_info(xai_results)")
    columns = [row[1] for row in cursor.fetchall()]
    
    # Build SELECT clause with existing columns
    metric_cols = ['fidelity', 'sparsity', 'complexity', 'reliability_correlation',
                   'reliability_ped', 'reliability_topk_overlap', 'reliability_kendall_tau',
                   'efficiency_error', 'computation_time']
    
    select_metrics = [f'AVG({col}) as {col}' for col in metric_cols if col in columns]
    
    query = f'''
        SELECT primary_use, option_number, model_name, xai_method,
               {', '.join(select_metrics)},
               COUNT(*) as n_samples
        FROM xai_results
    '''
    
    conditions = []
    params = []
    
    if primary_use:
        conditions.append('primary_use = ?')
        params.append(primary_use)
    if option_number is not None:
        conditions.append('option_number = ?')
        params.append(option_number)
    
    if conditions:
        query += ' WHERE ' + ' AND '.join(conditions)
    
    query += ' GROUP BY primary_use, option_number, model_name, xai_method'
    query += ' ORDER BY model_name, xai_method'
    
    df = pd.read_sql_query(query, conn, params=params)
    conn.close()
    return df


def get_available_datasets(db_path=XAI_DB):
    """Get available datasets from database."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT DISTINCT primary_use, option_number, COUNT(*) as n_results
        FROM xai_results
        GROUP BY primary_use, option_number
        ORDER BY primary_use, option_number
    ''')
    
    datasets = [{'primary_use': r[0], 'option_number': r[1], 'n_results': r[2]} 
                for r in cursor.fetchall()]
    conn.close()
    return datasets


def get_available_primary_uses(db_path=XAI_DB):
    """Get available primary uses with aggregated stats."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT primary_use, 
               COUNT(*) as n_results,
               COUNT(DISTINCT model_name) as n_models,
               COUNT(DISTINCT xai_method) as n_methods
        FROM xai_results
        GROUP BY primary_use
        ORDER BY primary_use
    ''')
    
    primary_uses = [{'primary_use': r[0], 'n_results': r[1], 'n_models': r[2], 'n_methods': r[3]} 
                    for r in cursor.fetchall()]
    conn.close()
    return primary_uses


def save_all_metrics_to_log(metrics=None):
    """Save all metric data as formatted tables to a single .log file (by primary_use)."""
    df = load_xai_results_by_primary_use()
    
    if len(df) == 0:
        print("❌ No data found in database!")
        return
    
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    log_filename = "all_metrics_by_primary_use.log"
    log_path = output_dir / log_filename
    
    if metrics is None:
        metrics = list(METRICS_CONFIG.keys())
    
    # Filter to only metrics that exist in the dataframe
    available_metrics = [m for m in metrics if METRICS_CONFIG[m]['column'] in df.columns]
    
    with open(log_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("XAI METRIC VALUES - BY PRIMARY USE AND XAI METHOD\n")
        f.write("(Aggregated across all models)\n")
        f.write("="*80 + "\n\n")
        
        for metric_key in available_metrics:
            if metric_key not in METRICS_CONFIG:
                continue
            
            config = METRICS_CONFIG[metric_key]
            column = config['column']
            
            plot_df = df[df[column].notna()].copy()
            
            if len(plot_df) == 0:
                continue
            
            f.write(f"\n{config['title']}\n")
            f.write("-" * 80 + "\n")
            
            primary_uses = sorted(plot_df['primary_use'].unique())
            methods = sorted(plot_df['xai_method'].unique())
            
            # Create pivot table
            pivot_df = plot_df.pivot_table(
                index='primary_use',
                columns='xai_method',
                values=column,
                aggfunc='first'
            )
            
            # Reorder columns and rows
            pivot_df = pivot_df.reindex(columns=[m for m in methods if m in pivot_df.columns])
            pivot_df = pivot_df.reindex(index=[p for p in primary_uses if p in pivot_df.index])
            
            # Rename index to display labels (without line breaks for log file)
            pivot_df.index = [get_display_label(p).replace('\n', ' ') for p in pivot_df.index]
            
            # Format to 3 decimal places
            formatted_df = pivot_df.applymap(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
            
            f.write(formatted_df.to_string())
            f.write("\n\n")
    
    print(f"✅ Saved all metrics to: {log_path}")
    return str(log_path)


# ============================
# LEGEND CREATION
# ============================
def create_legend_pdf(methods, output_path):

    fig = plt.figure(figsize=(1, 1))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')

    patches = []
    labels = []

    for method in methods:
        color = METHOD_COLORS.get(method, '#888888')
        label = method.upper() if method in ['tde', 'lime'] else method.replace('_', ' ').title()
        patches.append(
            plt.Rectangle((0, 0), 1, 1,
                          fc=color,
                          ec=BAR_EDGE_COLOR,
                          linewidth=BAR_EDGE_WIDTH)
        )
        labels.append(label)

    legend = ax.legend(
        patches,
        labels,
        loc='upper left',
        bbox_to_anchor=(0, 1),
        ncol=8,
        fontsize=LEGEND_FONT_SIZE,
        frameon=True,
        fancybox=False,
        handlelength=1.2,
        handleheight=1.0,
        handletextpad=0.35,
        columnspacing=0.7,
        borderpad=0.2,
        labelspacing=0.2
    )

    frame = legend.get_frame()
    frame.set_linewidth(1.0)
    frame.set_edgecolor('black')

    fig.canvas.draw()

    bbox = legend.get_window_extent(renderer=fig.canvas.get_renderer())
    bbox = bbox.transformed(fig.dpi_scale_trans.inverted())

    # Manual micro trim
    trim_left = 0.02
    trim_top = 0.02

    fig.set_size_inches(
        bbox.width - trim_left,
        bbox.height - trim_top
    )

    plt.savefig(
        output_path,
        format='pdf',
        bbox_inches='tight',
        pad_inches=0
    )

    plt.close(fig)

    return True


# ============================
# PLOTTING FUNCTIONS
# ============================
def create_metric_plot_by_primary_use(df, metric_key, output_path):
    """Create a single grouped bar plot for one metric with primary_use on x-axis."""
    if metric_key not in METRICS_CONFIG:
        print(f"  ⚠️ Unknown metric: {metric_key}")
        return False
    
    config = METRICS_CONFIG[metric_key]
    column = config['column']
    
    # Check if column exists in dataframe
    if column not in df.columns:
        print(f"  ⚠️ Column '{column}' not found in data (metric may not be in database yet)")
        return False
    
    plot_df = df[df[column].notna()].copy()
    
    if len(plot_df) == 0:
        print(f"  ⚠️ No data for metric: {metric_key}")
        return False
    
    # Get unique primary_uses and methods
    primary_uses = sorted(plot_df['primary_use'].unique())
    methods = sorted(plot_df['xai_method'].unique())
    
    n_primary_uses = len(primary_uses)
    n_methods = len(methods)
    
    if n_primary_uses == 0 or n_methods == 0:
        print(f"  ⚠️ No primary_uses or methods for metric: {metric_key}")
        return False
    
    # Create figure
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
    
    # Calculate bar positions
    x = np.arange(n_primary_uses)
    total_width = BAR_WIDTH * n_methods
    offsets = np.linspace(-total_width/2 + BAR_WIDTH/2, total_width/2 - BAR_WIDTH/2, n_methods)
    
    # Plot bars for each method
    for i, method in enumerate(methods):
        method_data = plot_df[plot_df['xai_method'] == method]
        
        values = []
        for primary_use in primary_uses:
            use_row = method_data[method_data['primary_use'] == primary_use]
            if len(use_row) > 0:
                val = use_row[column].values[0]
                values.append(val if pd.notna(val) else 0)
            else:
                values.append(0)
        
        if config.get('log_scale', False):
            values = [v if v > 0 else 1e-6 for v in values]
        
        color = METHOD_COLORS.get(method, '#888888')
        
        ax.bar(x + offsets[i], values, BAR_WIDTH,
               color=color, edgecolor=BAR_EDGE_COLOR,
               linewidth=BAR_EDGE_WIDTH, alpha=BAR_ALPHA)
    
    # Format x-axis labels using the mapping
    x_labels = [get_display_label(p) for p in primary_uses]
    
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=X_TICK_FONT_SIZE)
    ax.tick_params(axis='y', labelsize=Y_TICK_FONT_SIZE)
    
    # Set x-axis limits with separate left and right margins
    x_min = x[0] - X_MARGIN_LEFT
    x_max = x[-1] + X_MARGIN_RIGHT
    ax.set_xlim(x_min, x_max)
    
    if config.get('log_scale', False):
        ax.set_yscale('log')
    
    if SHOW_TITLE:
        ax.set_title(config['title'], fontsize=TITLE_FONT_SIZE, fontweight='bold', pad=10)
    
    # X-axis label is disabled by default (SHOW_X_AXIS_LABEL = False)
    if SHOW_X_AXIS_LABEL:
        ax.set_xlabel('Building Type', fontsize=AXIS_LABEL_FONT_SIZE)
    
    if SHOW_Y_AXIS_LABEL:
        ax.set_ylabel(config['ylabel'], fontsize=AXIS_LABEL_FONT_SIZE)
    
    if SHOW_GRID:
        ax.grid(True, alpha=GRID_ALPHA, axis=GRID_AXIS)
        ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    return True


def create_all_metric_plots(metrics=None):
    """Create PDF plots for all specified metrics with primary_use on x-axis."""
    df = load_xai_results_by_primary_use()
    
    if len(df) == 0:
        print("❌ No data found in database!")
        return []
    
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if metrics is None:
        metrics = list(METRICS_CONFIG.keys())
    
    # Filter to only metrics that exist in the dataframe
    available_metrics = [m for m in metrics if METRICS_CONFIG[m]['column'] in df.columns]
    unavailable_metrics = [m for m in metrics if METRICS_CONFIG[m]['column'] not in df.columns]
    
    methods = sorted(df['xai_method'].unique())
    primary_uses = sorted(df['primary_use'].unique())
    
    # Show display labels in console output
    display_labels = [get_display_label(p).replace('\n', ' ') for p in primary_uses]
    
    print(f"\n{'='*60}")
    print(f"📊 Creating XAI Metric Plots (by Primary Use)")
    print(f"{'='*60}")
    print(f"  Primary Uses: {primary_uses}")
    print(f"  Display Labels: {display_labels}")
    print(f"  Methods: {methods}")
    print(f"  Available Metrics ({len(available_metrics)}): {available_metrics}")
    if unavailable_metrics:
        print(f"  ⚠️  Unavailable Metrics ({len(unavailable_metrics)}): {unavailable_metrics}")
    print(f"  Output: {output_dir}")
    print(f"{'='*60}")
    
    created = []
    
    # Save metrics to log file
    log_file = save_all_metrics_to_log(available_metrics)
    if log_file:
        created.append(log_file)
    
    # Create legend
    legend_path = output_dir / "legend.pdf"
    print(f"\n  📋 Creating legend...")
    if create_legend_pdf(methods, legend_path):
        created.append(str(legend_path))
        print(f"     ✅ Saved: {legend_path}")
    
    # Create metric plots
    for metric_key in available_metrics:
        output_path = output_dir / f"{metric_key}_by_primary_use.pdf"
        print(f"\n  📈 Creating {metric_key} plot...")
        
        if create_metric_plot_by_primary_use(df, metric_key, output_path):
            created.append(str(output_path))
            print(f"     ✅ Saved: {output_path}")
        else:
            print(f"     ⚠️ Skipped: {metric_key}")
    
    print(f"\n{'='*60}")
    print(f"✅ Created {len(created)} files in {output_dir}")
    print(f"{'='*60}")
    
    return created


# ============================
# MAIN
# ============================
def main():
    """Main entry point - auto-generates all plots with primary_use on x-axis."""
    print("\n" + "="*60)
    print("📊 XAI Metric Visualization (by Primary Use)")
    print("="*60)
    
    # Check database schema
    db_info = check_database_columns()
    if db_info['missing_metrics']:
        print(f"\n⚠️  Missing metrics in database: {', '.join(db_info['missing_metrics'])}")
        print(f"    (These metrics will be skipped)")
    if db_info['existing_metrics']:
        print(f"✅ Available metrics: {', '.join(db_info['existing_metrics'])}")
    
    primary_uses = get_available_primary_uses()
    
    if not primary_uses:
        print("❌ No XAI results found in database!")
        return
    
    print(f"\nFound {len(primary_uses)} primary use(s):")
    for pu in primary_uses:
        display_label = get_display_label(pu['primary_use']).replace('\n', ' ')
        print(f"  - {pu['primary_use']} → {display_label}: {pu['n_results']} results, {pu['n_models']} models, {pu['n_methods']} methods")
    
    created = create_all_metric_plots()
    
    print(f"\n✅ Done! Created {len(created)} files in {OUTPUT_DIR}")
    return created


# ============================
# RUN
# ============================
if __name__ == "__main__":
    main()


📊 XAI Metric Visualization (by Primary Use)
✅ Available metrics: complexity, computation_time, efficiency_error, fidelity, reliability_correlation, reliability_kendall_tau, reliability_ped, reliability_topk_overlap, sparsity

Found 5 primary use(s):
  - health → Medical Clinic: 800 results, 10 models, 8 methods
  - office → Office Building: 800 results, 10 models, 8 methods
  - residential → Residential: 800 results, 10 models, 8 methods
  - retail → Retail Store: 800 results, 10 models, 8 methods
  - warehouse → Manufacturing Facility: 800 results, 10 models, 8 methods

📊 Creating XAI Metric Plots (by Primary Use)
  Primary Uses: ['health', 'office', 'residential', 'retail', 'warehouse']
  Display Labels: ['Medical Clinic', 'Office Building', 'Residential', 'Retail Store', 'Manufacturing Facility']
  Methods: ['deep', 'fastshap', 'gradient', 'lime', 'partition', 'permutation', 'sampling', 'tde']
  Available Metrics (9): ['fidelity', 'sparsity', 'complexity', 'reliability_correlation'

# statistical analysis

In [ ]:
# ============================
# LIBRARY IMPORTS
# ============================
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
import logging
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ============================
# CONFIGURATION
# ============================
XAI_DB = "xai_results.db"
OUTPUT_DIR = "./results/metrics"

REFERENCE_METHOD = 'tde'
SIGNIFICANCE_LEVEL = 0.05

# P-value correction options: 'bonferroni', 'holm', 'fdr_bh', 'fdr_by', 'none'
P_VALUE_CORRECTION = 'bonferroni'

TABLE_FONT_SIZE, TABLE_HEADER_FONT_SIZE = 10, 12

DECIMAL_PLACES = {
    'fidelity': {'mean': 1, 'std': 1, 'p_value': 3},
    'sparsity': {'mean': 1, 'std': 1, 'p_value': 3},
    'complexity': {'mean': 2, 'std': 2, 'p_value': 3},
    'reliability_correlation': {'mean': 1, 'std': 1, 'p_value': 3},
    'reliability_topk': {'mean': 1, 'std': 1, 'p_value': 3},
    'reliability_ped': {'mean': 2, 'std': 2, 'p_value': 3},
    'reliability_kendall': {'mean': 1, 'std': 1, 'p_value': 3},
    'efficiency_error': {'mean': 2, 'std': 2, 'p_value': 3},
    'computation_time': {'mean': 2, 'std': 2, 'p_value': 3},
}

METRICS_CONFIG = {
    'fidelity': {'column': 'fidelity', 'title': 'Fidelity', 'higher_better': True, 'display_scale': 100},
    'sparsity': {'column': 'sparsity', 'title': 'Sparsity', 'higher_better': True, 'display_scale': 1},
    'complexity': {'column': 'complexity', 'title': 'Complexity', 'higher_better': False, 'display_scale': 1},
    'reliability_correlation': {'column': 'reliability_correlation', 'title': 'Reliability (Pearson)', 'higher_better': True, 'display_scale': 100},
    'reliability_topk': {'column': 'reliability_topk_overlap', 'title': 'Reliability (Top-K)', 'higher_better': True, 'display_scale': 1},
    'reliability_ped': {'column': 'reliability_ped', 'title': 'Reliability (PED)', 'higher_better': False, 'display_scale': 1},
    'reliability_kendall': {'column': 'reliability_kendall_tau', 'title': 'Reliability (Kendall)', 'higher_better': True, 'display_scale': 100},
    'efficiency_error': {'column': 'efficiency_error', 'title': 'Efficiency Error', 'higher_better': False, 'display_scale': 1},
    'computation_time': {'column': 'computation_time', 'title': 'Computation Time', 'higher_better': False, 'display_scale': 1},
}


# ============================
# LOGGING SETUP
# ============================
def setup_logger(output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / "xai_analysis.log"
    
    logger = logging.getLogger('XAI_Analysis')
    logger.setLevel(logging.INFO)
    logger.handlers = []
    
    for handler in [logging.FileHandler(log_path, mode='w', encoding='utf-8'), logging.StreamHandler()]:
        handler.setFormatter(logging.Formatter('%(message)s'))
        logger.addHandler(handler)
    
    return logger, log_path


# ============================
# DATA LOADING
# ============================
def load_xai_results_raw(primary_use=None, option_number=None, db_path=XAI_DB):
    conn = sqlite3.connect(db_path)
    query = '''SELECT primary_use, option_number, model_name, sample_idx, xai_method,
               fidelity, sparsity, complexity, reliability_correlation, reliability_topk_overlap, 
               reliability_ped, reliability_kendall_tau, efficiency_error, computation_time
               FROM xai_results'''
    conditions, params = [], []
    if primary_use:
        conditions.append('primary_use = ?')
        params.append(primary_use)
    if option_number is not None:
        conditions.append('option_number = ?')
        params.append(option_number)
    if conditions:
        query += ' WHERE ' + ' AND '.join(conditions)
    
    df = pd.read_sql_query(query, conn, params=params)
    conn.close()
    return df


# ============================
# STATISTICAL TESTS
# ============================
def perform_paired_tests(df_raw, metric_column, method1, method2, model_name):
    df1 = df_raw[(df_raw['xai_method'] == method1) & (df_raw['model_name'] == model_name)][['sample_idx', metric_column]].dropna()
    df2 = df_raw[(df_raw['xai_method'] == method2) & (df_raw['model_name'] == model_name)][['sample_idx', metric_column]].dropna()
    merged = df1.merge(df2, on='sample_idx', suffixes=('_1', '_2'))
    
    if len(merged) < 3:
        return None, None, 0
    try:
        _, p_ttest = stats.ttest_rel(merged[f'{metric_column}_1'].values, merged[f'{metric_column}_2'].values)
        return p_ttest, len(merged)
    except:
        return None, 0


def compute_statistical_comparison(df_raw, metric_key):
    config = METRICS_CONFIG[metric_key]
    column, higher_better = config['column'], config['higher_better']
    models = sorted(df_raw['model_name'].unique())
    methods = sorted([m for m in df_raw['xai_method'].unique() if m != REFERENCE_METHOD])
    
    results = []
    for model in models:
        ref_data = df_raw[(df_raw['xai_method'] == REFERENCE_METHOD) & (df_raw['model_name'] == model)][column].dropna()
        if len(ref_data) == 0:
            continue
        ref_mean, ref_std = ref_data.mean(), ref_data.std()
        
        for method in methods:
            comp_data = df_raw[(df_raw['xai_method'] == method) & (df_raw['model_name'] == model)][column].dropna()
            if len(comp_data) == 0:
                continue
            comp_mean, comp_std = comp_data.mean(), comp_data.std()
            p_ttest, n_pairs = perform_paired_tests(df_raw, column, REFERENCE_METHOD, method, model)
            tde_better = (ref_mean > comp_mean) if higher_better else (ref_mean < comp_mean)
            
            results.append({
                'model': model, 'method': method, 
                'tde_mean': ref_mean, 'tde_std': ref_std,
                'comp_mean': comp_mean, 'comp_std': comp_std, 
                'p_ttest': p_ttest, 'n_pairs': n_pairs, 'tde_better': tde_better
            })
    
    if not results:
        return None
    
    results_df = pd.DataFrame(results)
    valid_mask = results_df['p_ttest'].notna()
    
    if valid_mask.sum() > 0 and P_VALUE_CORRECTION.lower() != 'none':
        _, corrected_p, _, _ = multipletests(results_df.loc[valid_mask, 'p_ttest'].values, 
                                              method=P_VALUE_CORRECTION, alpha=SIGNIFICANCE_LEVEL)
        results_df.loc[valid_mask, 'p_ttest_corr'] = corrected_p
    else:
        results_df['p_ttest_corr'] = results_df['p_ttest']
    
    results_df['sig_ttest'] = results_df['p_ttest_corr'] < SIGNIFICANCE_LEVEL
    return results_df


# ============================
# FORMATTING HELPERS
# ============================
def format_p_value(p, decimals=3):
    if p is None or np.isnan(p):
        return '-'
    return '<.001' if p < 0.001 else f'{p:.{decimals}f}'


def get_significance_symbol(p, significant, tde_better):
    if p is None or np.isnan(p) or not significant:
        return '', ''
    stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    return stars, '✓' if tde_better else '✗'


# ============================
# TEXT LOG TABLE CREATION
# ============================
def log_metric_table(df_raw, metric_key, logger):
    config = METRICS_CONFIG[metric_key]
    column, higher_better = config['column'], config['higher_better']
    scale_factor = config.get('display_scale', 1)  # Get scaling factor
    decimals = DECIMAL_PLACES.get(metric_key, {'mean': 2, 'std': 2, 'p_value': 3})
    mean_dec, std_dec, p_dec = decimals['mean'], decimals['std'], decimals['p_value']
    
    stats_df = compute_statistical_comparison(df_raw, metric_key)
    if stats_df is None or len(stats_df) == 0:
        logger.info(f"\n[No data for {metric_key}]")
        return
    
    models, methods = sorted(stats_df['model'].unique()), sorted(stats_df['method'].unique())
    
    direction = "↑Higher=Better" if higher_better else "↓Lower=Better"
    scale_note = f" (×{scale_factor})" if scale_factor != 1 else ""
    logger.info(f"\n{'='*120}")
    logger.info(f"{config['title'].upper()}{scale_note}: Paired t-test ({direction} | {P_VALUE_CORRECTION} | *p<.05 **p<.01 ***p<.001)")
    logger.info(f"{'='*120}")
    
    col_widths = [12, 18] + [18] * len(methods)
    header = ['Model'.ljust(col_widths[0]), 'TDE (Ref)'.ljust(col_widths[1])]
    header += [((m.upper() if m == 'lime' else m.replace('_', ' ').title()).ljust(col_widths[i+2])) for i, m in enumerate(methods)]
    logger.info(''.join(header))
    logger.info('-' * 120)
    
    for model in models:
        tde_data = df_raw[(df_raw['xai_method'] == REFERENCE_METHOD) & (df_raw['model_name'] == model)][column].dropna()
        # Apply scaling factor to TDE values
        tde_str = f'{tde_data.mean()*scale_factor:.{mean_dec}f}±{tde_data.std()*scale_factor:.{std_dec}f}' if len(tde_data) > 0 else '-'
        
        line1 = [model.ljust(col_widths[0]), tde_str.ljust(col_widths[1])]
        line2 = [''.ljust(col_widths[0]), ''.ljust(col_widths[1])]
        
        for i, method in enumerate(methods):
            ms = stats_df[(stats_df['model'] == model) & (stats_df['method'] == method)]
            if len(ms) == 0:
                line1.append('-'.ljust(col_widths[i+2]))
                line2.append(''.ljust(col_widths[i+2]))
                continue
            ms = ms.iloc[0]
            # Apply scaling factor to comparison values
            line1.append(f'{ms["comp_mean"]*scale_factor:.{mean_dec}f}±{ms["comp_std"]*scale_factor:.{std_dec}f}'.ljust(col_widths[i+2]))
            stars, symbol = get_significance_symbol(ms['p_ttest_corr'], ms['sig_ttest'], ms['tde_better'])
            p_str = format_p_value(ms['p_ttest_corr'], p_dec)
            line2.append((f'p={p_str}{stars} {symbol}' if ms['sig_ttest'] else f'p={p_str}').ljust(col_widths[i+2]))
        
        logger.info(''.join(line1))
        logger.info(''.join(line2))
        logger.info('')
    
    logger.info('Blue=TDE(ref) | Green=TDE better (✓) | Red=TDE worse (✗)\n')


def log_summary_table(df_raw, logger):
    logger.info(f"\n{'='*80}")
    logger.info("TDE PERFORMANCE SUMMARY - Paired t-test (p<0.05)")
    logger.info(f"{'='*80}")
    logger.info(f"{'Metric':<30} {'Wins ✓':>10} {'Losses ✗':>12} {'Ties':>10} {'Win%':>10}")
    logger.info('-' * 80)
    
    summary_data = []
    for metric_key, config in METRICS_CONFIG.items():
        stats_df = compute_statistical_comparison(df_raw, metric_key)
        if stats_df is None:
            continue
        wins = stats_df[stats_df['sig_ttest'] & stats_df['tde_better']].shape[0]
        losses = stats_df[stats_df['sig_ttest'] & ~stats_df['tde_better']].shape[0]
        ties = stats_df[~stats_df['sig_ttest']].shape[0]
        total = len(stats_df)
        win_pct = f'{wins/total*100:.1f}%' if total > 0 else '-'
        logger.info(f"{config['title']:<30} {wins:>10} {losses:>12} {ties:>10} {win_pct:>10}")
        summary_data.append((wins, losses, ties))
    
    if summary_data:
        total_wins, total_losses, total_ties = sum(d[0] for d in summary_data), sum(d[1] for d in summary_data), sum(d[2] for d in summary_data)
        grand_total = total_wins + total_losses + total_ties
        logger.info('-' * 80)
        logger.info(f"{'TOTAL':<30} {total_wins:>10} {total_losses:>12} {total_ties:>10} {f'{total_wins/grand_total*100:.1f}%' if grand_total > 0 else '-':>10}")
    logger.info('=' * 80)


# ============================
# PDF TABLE CREATION
# ============================
def create_metric_table_page(fig, df_raw, metric_key, page_num, total_pages):
    config = METRICS_CONFIG[metric_key]
    column, higher_better = config['column'], config['higher_better']
    scale_factor = config.get('display_scale', 1)  # Get scaling factor
    decimals = DECIMAL_PLACES.get(metric_key, {'mean': 2, 'std': 2, 'p_value': 3})
    mean_dec, std_dec, p_dec = decimals['mean'], decimals['std'], decimals['p_value']
    
    stats_df = compute_statistical_comparison(df_raw, metric_key)
    if stats_df is None or len(stats_df) == 0:
        return False
    
    models, methods = sorted(stats_df['model'].unique()), sorted(stats_df['method'].unique())
    
    table_data, col_labels = [], ['Model', 'TDE (Ref)'] + [m.upper() if m == 'lime' else m.replace('_', ' ').title() for m in methods]
    
    for model in models:
        row = [model]
        tde_data = df_raw[(df_raw['xai_method'] == REFERENCE_METHOD) & (df_raw['model_name'] == model)][column].dropna()
        # Apply scaling factor to TDE values
        row.append(f'{tde_data.mean()*scale_factor:.{mean_dec}f}±{tde_data.std()*scale_factor:.{std_dec}f}' if len(tde_data) > 0 else '-')
        
        for method in methods:
            ms = stats_df[(stats_df['model'] == model) & (stats_df['method'] == method)]
            if len(ms) == 0:
                row.append('-')
                continue
            ms = ms.iloc[0]
            stars, symbol = get_significance_symbol(ms['p_ttest_corr'], ms['sig_ttest'], ms['tde_better'])
            p_str = format_p_value(ms['p_ttest_corr'], p_dec)
            # Apply scaling factor to comparison values
            line1 = f'{ms["comp_mean"]*scale_factor:.{mean_dec}f}±{ms["comp_std"]*scale_factor:.{std_dec}f}'
            line2 = f'p={p_str}{stars} {symbol}' if ms['sig_ttest'] else f'p={p_str}'
            row.append(f'{line1}\n{line2}')
        table_data.append(row)
    
    ax = fig.add_subplot(111)
    ax.axis('off')
    
    table = ax.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(TABLE_FONT_SIZE)
    table.scale(1.2, 2.2)
    
    n_cols, n_rows = len(col_labels), len(table_data)
    for j in range(n_cols):
        table[(0, j)].set_text_props(weight='bold', fontsize=TABLE_HEADER_FONT_SIZE)
        table[(0, j)].set_facecolor('#E6E6E6')
    
    for i in range(1, n_rows + 1):
        table[(i, 1)].set_facecolor('#E3F2FD')
        table[(i, 0)].set_text_props(weight='bold')
        table[(i, 0)].set_facecolor('#F5F5F5')
    
    for i, model in enumerate(models):
        for j, method in enumerate(methods):
            ms = stats_df[(stats_df['model'] == model) & (stats_df['method'] == method)]
            if len(ms) > 0 and ms.iloc[0]['sig_ttest']:
                table[(i + 1, j + 2)].set_facecolor('#E8F5E9' if ms.iloc[0]['tde_better'] else '#FFEBEE')
    
    direction = "↑Higher=Better" if higher_better else "↓Lower=Better"
    scale_note = f" (×{scale_factor})" if scale_factor != 1 else ""
    ax.set_title(f"{config['title']}{scale_note}: Paired t-test ({direction} | {P_VALUE_CORRECTION} | *p<.05 **p<.01 ***p<.001)", 
                 fontsize=TABLE_HEADER_FONT_SIZE, fontweight='bold', pad=5)
    fig.text(0.5, 0.01, f"Blue=TDE(ref) | Green=TDE better (✓) | Red=TDE worse (✗) | Page {page_num}/{total_pages}", 
             ha='center', fontsize=TABLE_FONT_SIZE-1, style='italic')
    return True


def create_summary_table_page(fig, df_raw, page_num, total_pages):
    summary_data = []
    for metric_key, config in METRICS_CONFIG.items():
        stats_df = compute_statistical_comparison(df_raw, metric_key)
        if stats_df is None:
            continue
        wins = stats_df[stats_df['sig_ttest'] & stats_df['tde_better']].shape[0]
        losses = stats_df[stats_df['sig_ttest'] & ~stats_df['tde_better']].shape[0]
        ties = stats_df[~stats_df['sig_ttest']].shape[0]
        total = len(stats_df)
        summary_data.append({'Metric': config['title'], 'Wins ✓': wins, 'Losses ✗': losses, 
                            'Ties': ties, 'Win%': f'{wins/total*100:.1f}%' if total > 0 else '-'})
    
    if not summary_data:
        return False
    
    summary_df = pd.DataFrame(summary_data)
    ax = fig.add_subplot(111)
    ax.axis('off')
    
    table = ax.table(cellText=summary_df.values, colLabels=summary_df.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(TABLE_FONT_SIZE)
    table.scale(1.2, 1.8)
    
    for j in range(len(summary_df.columns)):
        table[(0, j)].set_text_props(weight='bold', fontsize=TABLE_HEADER_FONT_SIZE)
        table[(0, j)].set_facecolor('#E6E6E6')
    
    ax.set_title('TDE Performance Summary - Paired t-test (p<0.05)', fontsize=TABLE_HEADER_FONT_SIZE, fontweight='bold', pad=5)
    fig.text(0.5, 0.01, f"Page {page_num}/{total_pages}", ha='center', fontsize=TABLE_FONT_SIZE-1, style='italic')
    return True


# ============================
# MAIN PDF CREATION
# ============================
def create_combined_pdf(df_raw, output_path, metrics=None):
    if metrics is None:
        metrics = list(METRICS_CONFIG.keys())
    
    total_pages, page_num = len(metrics) + 1, 1
    n_models = len(df_raw['model_name'].unique())
    table_height = max(4, n_models * 0.6 + 1.5)
    
    with PdfPages(output_path) as pdf:
        for metric_key in metrics:
            fig = plt.figure(figsize=(14, table_height))
            if create_metric_table_page(fig, df_raw, metric_key, page_num, total_pages):
                plt.tight_layout(pad=0.5)
                pdf.savefig(fig, bbox_inches='tight')
                page_num += 1
            plt.close(fig)
        
        fig = plt.figure(figsize=(14, max(3, len(metrics) * 0.5 + 1.5)))
        if create_summary_table_page(fig, df_raw, page_num, total_pages):
            plt.tight_layout(pad=0.5)
            pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
    return True


# ============================
# MAIN EXECUTION
# ============================
def create_all_outputs(primary_use=None, option_number=None, metrics=None):
    output_dir = Path(OUTPUT_DIR)
    logger, log_path = setup_logger(output_dir)
    
    df_raw = load_xai_results_raw(primary_use, option_number)
    if len(df_raw) == 0:
        logger.info("❌ No data found!")
        return []
    
    if metrics is None:
        metrics = list(METRICS_CONFIG.keys())
    
    methods, models = sorted(df_raw['xai_method'].unique()), sorted(df_raw['model_name'].unique())
    
    logger.info("=" * 70)
    logger.info("XAI STATISTICAL ANALYSIS")
    logger.info("=" * 70)
    logger.info(f"Models: {models}")
    logger.info(f"Methods: {methods}")
    logger.info(f"Reference: {REFERENCE_METHOD.upper()}")
    logger.info(f"Test: Paired t-test | P-correction: {P_VALUE_CORRECTION} | α={SIGNIFICANCE_LEVEL}")
    logger.info("=" * 70)
    
    for metric_key in metrics:
        log_metric_table(df_raw, metric_key, logger)
    log_summary_table(df_raw, logger)
    
    pdf_path = output_dir / "xai_statistical_results.pdf"
    if create_combined_pdf(df_raw, pdf_path, metrics):
        logger.info(f"\n✅ Created: {pdf_path}")
        logger.info(f"   Pages: {len(metrics) + 1} (1 per metric + summary)")
    
    logger.info(f"\n📄 Log saved: {log_path}")
    logger.info("=" * 70)
    return [pdf_path, log_path]


def main():
    create_all_outputs()


if __name__ == "__main__":
    main()

XAI STATISTICAL ANALYSIS
Models: ['BGRU', 'BLSTM', 'CNN1D', 'DCNN', 'GRU', 'LSTM', 'TCN', 'TFT', 'TST', 'WaveNet']
Methods: ['deep', 'fastshap', 'gradient', 'lime', 'partition', 'permutation', 'sampling', 'tde']
Reference: TDE
Test: Paired t-test | P-correction: fdr_bh | α=0.05

FIDELITY (×100): Paired t-test (↑Higher=Better | fdr_bh | *p<.05 **p<.01 ***p<.001)
Model       TDE (Ref)         Deep              Fastshap          Gradient          LIME              Partition         Permutation       Sampling          
------------------------------------------------------------------------------------------------------------------------
BGRU        6.7±5.3           7.5±5.4           1.1±1.0           7.6±5.4           4.9±5.0           1.3±1.5           7.0±4.9           6.8±5.1           
                              p=0.058           p=<.001*** ✓      p=0.048* ✗        p=0.002** ✓       p=<.001*** ✓      p=0.630           p=0.941           

BLSTM       6.7±5.2           7.2±5.2      

# Oblation

In [10]:
# ============================
# LIBRARY IMPORTS
# ============================
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
import logging

# ============================
# CONFIGURATION
# ============================
ABLATION_DB = "databases/ablation_studies.db"
OUTPUT_DIR = "./results/ablation"
ABLATION_TABLE = "ablation_results"

SIGNIFICANCE_LEVEL = 0.05
P_VALUE_CORRECTION = 'fdr_bh'  # Benjamini-Hochberg FDR correction

REFERENCE_VARIANTS = {
    'architecture': 'full_tde',
    'loss_terms': 'full_loss'
}

DECIMAL_PLACES = {
    'fidelity': 4, 'sparsity': 2, 'complexity': 2, 'reliability_correlation': 4,
    'reliability_topk_overlap': 2, 'reliability_ped': 2, 'reliability_kendall_tau': 3,
    'completeness_error': 3, 'computation_time': 3, 'best_validation_loss': 6, 'training_time': 2
}

METRICS_CONFIG = {
    'fidelity': {'title': 'Fidelity', 'higher_better': True},
    'sparsity': {'title': 'Sparsity (%)', 'higher_better': True},
    'complexity': {'title': 'Complexity', 'higher_better': False},
    'reliability_correlation': {'title': 'Reliability (Corr)', 'higher_better': True},
    'reliability_topk_overlap': {'title': 'Reliability (Top-K)', 'higher_better': True},
    'reliability_ped': {'title': 'Reliability (PED)', 'higher_better': False},
    'reliability_kendall_tau': {'title': 'Reliability (Kendall)', 'higher_better': True},
    'completeness_error': {'title': 'Completeness Error', 'higher_better': False},
    'computation_time': {'title': 'Computation Time', 'higher_better': False},
    'best_validation_loss': {'title': 'Val Loss', 'higher_better': False},
    'training_time': {'title': 'Training Time', 'higher_better': False},
}

# ============================
# LOGGING SETUP
# ============================
def setup_logger(output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / "ablation_comparison.log"
    
    logger = logging.getLogger('Ablation_Comparison')
    logger.handlers = []
    logger.setLevel(logging.INFO)
    
    handler = logging.FileHandler(log_path, mode='w', encoding='utf-8')
    handler.setFormatter(logging.Formatter('%(message)s'))
    logger.addHandler(handler)
    logger.addHandler(logging.StreamHandler())
    
    return logger, log_path

# ============================
# DATA LOADING
# ============================
def load_ablation_results_raw(ablation_category, db_path=ABLATION_DB):
    """Load ablation results for a specific category."""
    conn = sqlite3.connect(db_path)
    query = f'''SELECT ablation_category, variant_name, sample_idx, 
               fidelity, sparsity, complexity, reliability_correlation, reliability_topk_overlap, 
               reliability_ped, reliability_kendall_tau, completeness_error, computation_time,
               best_validation_loss, training_time
               FROM {ABLATION_TABLE}
               WHERE ablation_category = ?'''
    df = pd.read_sql_query(query, conn, params=[ablation_category])
    conn.close()
    return df

# ============================
# STATISTICAL TESTS
# ============================
def compute_cohens_d(data1, data2):
    """Compute Cohen's d effect size."""
    n1, n2 = len(data1), len(data2)
    var1, var2 = np.var(data1, ddof=1), np.var(data2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    return (np.mean(data1) - np.mean(data2)) / pooled_std

def perform_paired_test(ref_df, comp_df, metric_key):
    """
    Perform paired t-test comparing reference variant to comparison variant.
    
    CRITICAL FIX: Uses sample_idx as the pairing key, not DataFrame index.
    Handles duplicate sample_idx by averaging values per sample.
    """
    # Group by sample_idx and take mean to handle duplicates
    ref_grouped = ref_df.groupby('sample_idx')[metric_key].mean()
    comp_grouped = comp_df.groupby('sample_idx')[metric_key].mean()
    
    # Find common sample_idx values
    common_samples = sorted(set(ref_grouped.index) & set(comp_grouped.index))
    
    if len(common_samples) < 2:
        return None, None, 0
    
    # Extract paired values - now guaranteed same length
    ref_paired = ref_grouped.loc[common_samples].values
    comp_paired = comp_grouped.loc[common_samples].values
    
    try:
        # Paired t-test
        t_stat, p_val = stats.ttest_rel(ref_paired, comp_paired)
        
        # Effect size
        effect_size = compute_cohens_d(ref_paired, comp_paired)
        
        return p_val, effect_size, len(common_samples)
    except Exception as e:
        print(f"Error in paired test: {e}")
        return None, None, 0

def get_effect_size_interpretation(d):
    """Interpret Cohen's d effect size."""
    abs_d = abs(d)
    if abs_d < 0.2:
        return "negligible"
    elif abs_d < 0.5:
        return "small"
    elif abs_d < 0.8:
        return "medium"
    else:
        return "large"

def get_significance_marker(p_val, ref_better, effect_size):
    """Generate significance marker with effect size indicator."""
    if p_val is None or p_val >= SIGNIFICANCE_LEVEL:
        return ""
    
    # Star significance levels
    if p_val < 0.001:
        stars = "***"
    elif p_val < 0.01:
        stars = "**"
    else:
        stars = "*"
    
    # Direction symbol
    symbol = "✓" if ref_better else "✗"
    
    # Effect size indicator
    effect_interp = get_effect_size_interpretation(effect_size)
    effect_abbrev = {'negligible': 'neg', 'small': 'sm', 'medium': 'md', 'large': 'lg'}[effect_interp]
    
    return f" {symbol}{stars}({effect_abbrev})"

# ============================
# TABLE CREATION
# ============================
def create_comparison_table(df, category, ref_variant, logger):
    """Create comprehensive comparison table with statistical tests."""
    metrics = list(METRICS_CONFIG.keys())
    variants = sorted([v for v in df['variant_name'].unique() if v != ref_variant])
    
    if len(variants) == 0:
        logger.info(f"⚠️ No comparison variants found for {category}")
        return ""
    
    # Extract reference data
    ref_df = df[df['variant_name'] == ref_variant].copy()
    
    if len(ref_df) == 0:
        logger.info(f"⚠️ No reference data for {ref_variant} in {category}")
        return ""
    
    # Collect p-values for multiple comparison correction
    all_p_values = []
    test_results = {}  # {(variant, metric): (p_val, effect_size, n_pairs, ref_better)}
    
    # Run all tests first
    for variant in variants:
        comp_df = df[df['variant_name'] == variant].copy()
        
        for metric_key in metrics:
            p_val, effect_size, n_pairs = perform_paired_test(ref_df, comp_df, metric_key)
            
            if p_val is not None and n_pairs >= 2:
                higher_better = METRICS_CONFIG[metric_key]['higher_better']
                
                ref_mean = ref_df[metric_key].mean()
                comp_mean = comp_df[metric_key].mean()
                ref_is_better = (ref_mean > comp_mean) if higher_better else (ref_mean < comp_mean)
                
                all_p_values.append(p_val)
                test_results[(variant, metric_key)] = (p_val, effect_size, n_pairs, ref_is_better)
    
    # Apply multiple comparison correction
    if len(all_p_values) > 0:
        _, corrected_p_values, _, _ = multipletests(all_p_values, alpha=SIGNIFICANCE_LEVEL, method=P_VALUE_CORRECTION)
        
        # Update test_results with corrected p-values
        corrected_results = {}
        for i, (key, (_, effect_size, n_pairs, ref_better)) in enumerate(test_results.items()):
            corrected_results[key] = (corrected_p_values[i], effect_size, n_pairs, ref_better)
        test_results = corrected_results
    
    # Build table
    table_lines = []
    table_lines.append("=" * 180)
    table_lines.append(f"{category.upper()} CATEGORY - Statistical Comparison vs {ref_variant.upper()}")
    table_lines.append(f"{'='*180}")
    table_lines.append(f"Legend: ✓={ref_variant} Better | ✗={ref_variant} Worse | *p<0.05 **p<0.01 ***p<0.001")
    table_lines.append(f"Effect Size: neg=negligible(<0.2) | sm=small(0.2-0.5) | md=medium(0.5-0.8) | lg=large(>0.8)")
    table_lines.append(f"P-values: FDR-corrected using {P_VALUE_CORRECTION}")
    table_lines.append("=" * 180)
    
    # Header
    header = f"{'Metric':<28} {ref_variant.upper():<25}"
    for variant in variants:
        header += f" {variant:<40}"
    table_lines.append(header)
    table_lines.append("-" * 180)
    
    # Compute reference statistics
    ref_stats = {}
    for metric_key in metrics:
        ref_data = ref_df[metric_key].dropna()
        if len(ref_data) > 0:
            ref_stats[metric_key] = {
                'mean': ref_data.mean(),
                'std': ref_data.std(),
                'n': len(ref_data)
            }
    
    # Rows
    for metric_key in metrics:
        metric_title = METRICS_CONFIG[metric_key]['title']
        decimals = DECIMAL_PLACES[metric_key]
        
        if metric_key not in ref_stats:
            continue
        
        ref_mean = ref_stats[metric_key]['mean']
        ref_std = ref_stats[metric_key]['std']
        ref_n = ref_stats[metric_key]['n']
        ref_str = f"{ref_mean:.{decimals}f}±{ref_std:.{decimals}f} (n={ref_n})"
        
        row = f"{metric_title:<28} {ref_str:<25}"
        
        for variant in variants:
            comp_df_variant = df[df['variant_name'] == variant].copy()
            comp_data = comp_df_variant[metric_key].dropna()
            
            if len(comp_data) == 0:
                row += f" {'-':<40}"
                continue
            
            comp_mean = comp_data.mean()
            comp_std = comp_data.std()
            comp_n = len(comp_data)
            
            # Get test results
            if (variant, metric_key) in test_results:
                p_val, effect_size, n_pairs, ref_better = test_results[(variant, metric_key)]
                marker = get_significance_marker(p_val, ref_better, effect_size)
                
                cell_str = f"{comp_mean:.{decimals}f}±{comp_std:.{decimals}f}{marker}"
                if p_val is not None:
                    cell_str += f" p={p_val:.4f}"
            else:
                cell_str = f"{comp_mean:.{decimals}f}±{comp_std:.{decimals}f} (no pairs)"
            
            row += f" {cell_str:<40}"
        
        table_lines.append(row)
    
    table_lines.append("=" * 180)
    
    # Summary statistics
    table_lines.append("")
    table_lines.append("SUMMARY:")
    
    for variant in variants:
        wins = sum(1 for (v, m), (p, _, _, ref_better) in test_results.items() 
                   if v == variant and p < SIGNIFICANCE_LEVEL and not ref_better)
        losses = sum(1 for (v, m), (p, _, _, ref_better) in test_results.items() 
                    if v == variant and p < SIGNIFICANCE_LEVEL and ref_better)
        ties = sum(1 for (v, m), (p, _, _, ref_better) in test_results.items() 
                   if v == variant and p >= SIGNIFICANCE_LEVEL)
        
        total_tests = wins + losses + ties
        
        if total_tests > 0:
            table_lines.append(f"  {variant}: {wins} wins, {losses} losses, {ties} ties (out of {total_tests} tests)")
    
    table_lines.append("=" * 180)
    table_lines.append("")
    
    return "\n".join(table_lines)

# ============================
# MAIN EXECUTION
# ============================
def main():
    output_dir = Path(OUTPUT_DIR)
    logger, log_path = setup_logger(output_dir)
    
    logger.info("=" * 180)
    logger.info("ABLATION STUDY STATISTICAL COMPARISON (FIXED VERSION)")
    logger.info("=" * 180)
    logger.info(f"Database: {ABLATION_DB}")
    logger.info(f"Test: Paired t-test (paired on sample_idx)")
    logger.info(f"P-correction: {P_VALUE_CORRECTION} | α={SIGNIFICANCE_LEVEL}")
    logger.info(f"Effect Size: Cohen's d")
    logger.info("=" * 180)
    logger.info("")
    
    for category, ref_variant in REFERENCE_VARIANTS.items():
        logger.info(f"\n{'='*180}")
        logger.info(f"Loading {category.upper()}...")
        logger.info(f"{'='*180}")
        
        df = load_ablation_results_raw(category)
        
        if len(df) == 0:
            logger.info(f"❌ No data for {category}")
            continue
        
        if ref_variant not in df['variant_name'].values:
            logger.info(f"❌ Reference variant '{ref_variant}' not found in {category}")
            logger.info(f"   Available variants: {df['variant_name'].unique().tolist()}")
            continue
        
        # Show data summary
        logger.info(f"\nData Summary:")
        logger.info(f"  Total rows: {len(df)}")
        for variant in df['variant_name'].unique():
            variant_df = df[df['variant_name'] == variant]
            n_samples = variant_df['sample_idx'].nunique()
            logger.info(f"  {variant}: {len(variant_df)} rows, {n_samples} unique samples")
        
        table_str = create_comparison_table(df, category, ref_variant, logger)
        logger.info(table_str)
    
    logger.info(f"\n{'='*180}")
    logger.info(f"📄 Complete results saved to: {log_path}")
    logger.info(f"{'='*180}")

if __name__ == "__main__":
    main()

ABLATION STUDY STATISTICAL COMPARISON (FIXED VERSION)
Database: databases/ablation_studies.db
Test: Paired t-test (paired on sample_idx)
P-correction: fdr_bh | α=0.05
Effect Size: Cohen's d


Loading ARCHITECTURE...

Data Summary:
  Total rows: 2465
  no_attention: 500 rows, 40 unique samples
  no_direct_input: 484 rows, 40 unique samples
  no_soft_threshold: 481 rows, 40 unique samples
  tcn_baseline: 500 rows, 40 unique samples
  full_tde: 500 rows, 40 unique samples
ARCHITECTURE CATEGORY - Statistical Comparison vs FULL_TDE
Legend: ✓=full_tde Better | ✗=full_tde Worse | *p<0.05 **p<0.01 ***p<0.001
Effect Size: neg=negligible(<0.2) | sm=small(0.2-0.5) | md=medium(0.5-0.8) | lg=large(>0.8)
P-values: FDR-corrected using fdr_bh
Metric                       FULL_TDE                  no_attention                             no_direct_input                          no_soft_threshold                        tcn_baseline                            
--------------------------------------------